# Seq2Seq 모델 Q&A + Chatbot 만들기

1. qna 데이터셋을 찾아서 처리해서 준비한다.( 전처리 전반)
2. 인코더디코더seq2seq(인코더+디코더) 모델 만들기
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습
4. 챗봇을 만든다(모델 추론 +while문)

In [2]:
# %pip install datasets

### BCCard-Finance-Kor-QnA
- 제작기관: BC Card
- 도메인: 금융 (카드, 결제, 금융서비스)
- 데이터 용량: 31250
- 목적: 금융 지식 기반 챗봇 및 금융 도메인 LLM 학습
- 특징: 금융 서비스 중심 질문(카드 결제, 발급, 금융 서비스, 제도, 연체 이자 등)


https://huggingface.co/datasets/BCCard/BCCard-Finance-Kor-QnA?utm_source=chatgpt.com&library=datasets

In [1]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv')
ds = df[['Q', 'A']]
ds

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.


In [2]:
ds

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.


In [4]:
import re
import pandas as pd
import numpy as np
from konlpy.tag import Okt
import torch
import torch.nn as nn
import torch.functional as F
from torch.utils.data import Dataset, DataLoader

In [5]:
# 전역변수
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 30000
EMBEDDING_DIM = 100
LATENT_DIM = 512

In [ ]:
import re
import pandas as pd
from collections import Counter
from konlpy.tag import Okt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 전역변수
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 30000
EMBEDDING_DIM = 100
LATENT_DIM = 512

okt = Okt()

# 토큰화 함수
def tokenizer(text):
    return okt.morphs(text)

# 반복 문자 정규화 함수
def normalize_repeats(text):
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ가-힣])\1{3,}', r'\1\1', text)
    text = re.sub(r'([ㅋㅎㅠㅜ])\1{2,}', r'\1\1', text)
    text = re.sub(r'([!?.,~])\1{2,}', r'\1\1', text)
    return text

# 텍스트 정제 함수
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = normalize_repeats(text)
    return text

# 토큰 리스트를 다시 문장으로 합치는 함수
def detokenize(tokens):
    sentence = " ".join(tokens)
    sentence = re.sub(r"\s+([?.!,~])", r"\1", sentence)
    sentence = re.sub(r"\(\s+", "(", sentence)
    sentence = re.sub(r"\s+\)", ")", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence

# 문장 하나 전처리
def preprocess_text(text):
    text = clean_text(text)
    tokens = tokenizer(text)
    return detokenize(tokens)


# DataFrame 전처리
def preprocess_data(df, q_col='Q', a_col='A'):
    df = df[[q_col, a_col]].copy()
    df = df.rename(columns={q_col: 'question', a_col: 'answer'})

    # 결측치 제거
    df = df.dropna(subset=['question', 'answer'])

    # 문자열화 + 공백 제거
    df['question'] = df['question'].astype(str).str.strip()
    df['answer'] = df['answer'].astype(str).str.strip()

    # 빈 문자열 제거
    df = df[(df['question'] != '') & (df['answer'] != '')].reset_index(drop=True)

    # 전처리
    df['question'] = df['question'].apply(preprocess_text)
    df['answer'] = df['answer'].apply(preprocess_text)

    return df[['question', 'answer']]


In [30]:
# 전처리
preprocessed_df = preprocess_data(df, q_col='Q', a_col='A')

# 확인
print(preprocessed_df.head())

# 기존 코드 재사용을 위해 dict 리스트로 변환
preprocessed_ds = preprocessed_df.to_dict(orient='records')

print(preprocessed_ds[0])
print(len(preprocessed_ds))

# 저장
preprocessed_df.to_csv('./preprocessed_data.csv', index=False, encoding='utf-8-sig')

             question        answer
0              12시 땡!   하루 가 또 가네요.
1        1 지망 학교 떨어졌어    위로 해 드립니다.
2     3 박 4일 놀러 가고 싶다  여행 은 언제나 좋죠.
3  3 박 4일 정도 놀러 가고 싶다  여행 은 언제나 좋죠.
4             PPL 심하네   눈살 이 찌푸려지죠.
{'question': '12시 땡!', 'answer': '하루 가 또 가네요.'}
11823


In [31]:
# vocab 만들기
special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
counter = Counter()

for item in preprocessed_ds:
    question_tokens = item['question'].split()
    answer_tokens = item['answer'].split()

    counter.update(question_tokens)
    counter.update(answer_tokens)

vocab = special_tokens + list(counter.keys())
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}  # 이 줄은 같이 수정

# 문장을 정수 시퀀스로 변환
def sentence_to_sequence(tokens, word2idx):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

tokenized_data = []

for sample in preprocessed_ds:
    question_tokens = sample['question'].split()
    answer_tokens = sample['answer'].split()

    q_ids = sentence_to_sequence(question_tokens, word2idx)
    a_ids = [word2idx['<SOS>']] + sentence_to_sequence(answer_tokens, word2idx) + [word2idx['<EOS>']]

    tokenized_data.append({
        'question_ids': q_ids,
        'answer_ids': a_ids
    })

print(tokenized_data[0])

# padding
max_q_len = max(len(sample['question_ids']) for sample in tokenized_data)
max_a_len = max(len(sample['answer_ids']) for sample in tokenized_data)

def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value] * (max_len - len(seq))

pad_idx = word2idx['<PAD>']
for sample in tokenized_data:
    sample['question_ids'] = pad_sequence(sample['question_ids'], max_q_len, pad_idx)
    sample['answer_ids'] = pad_sequence(sample['answer_ids'], max_a_len, pad_idx)

print(tokenized_data[0])

{'question_ids': [4, 5], 'answer_ids': [1, 6, 7, 8, 9, 2]}
{'question_ids': [4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'answer_ids': [1, 6, 7, 8, 9, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [32]:
class ChatDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        encoder_input = sample['question_ids']

        answer_ids = sample['answer_ids']
        decoder_input = answer_ids[:-1]
        decoder_target = answer_ids[1:]

        return {
            'encoder_input': torch.tensor(encoder_input, dtype=torch.long),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }

chat_dataset = ChatDataset(tokenized_data)
train_loader = DataLoader(chat_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(chat_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [33]:
# 문장을 정수 시퀀스
def sentence_to_sequence(tokens, word2idx):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

tokenized_data = []

for sample in preprocessed_ds:
    
    question_tokens = sample['question'].split()
    answer_tokens = sample['answer'].split()

    # 질문 문장
    q_ids = sentence_to_sequence(question_tokens, word2idx)

    # 답변 문장
    a_ids = [word2idx['<SOS>']] + sentence_to_sequence(answer_tokens, word2idx) + [word2idx['<EOS>']]

    tokenized_data.append({
        'question_ids': q_ids,
        'answer_ids': a_ids
    })

print(tokenized_data[0])

{'question_ids': [4, 5], 'answer_ids': [1, 6, 7, 8, 9, 2]}


In [34]:
# padding
max_q_len = max(len(sample['question_ids']) for sample in tokenized_data)
max_a_len = max(len(sample['answer_ids']) for sample in tokenized_data)

def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value] * (max_len - len(seq))

pad_idx = word2idx['<PAD>']
for sample in tokenized_data:
    sample['question_ids'] = pad_sequence(sample['question_ids'], max_q_len, pad_idx)
    sample['answer_ids'] = pad_sequence(sample['answer_ids'], max_a_len, pad_idx)
    
print((tokenized_data[0]))

{'question_ids': [4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'answer_ids': [1, 6, 7, 8, 9, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [35]:
# dataset/ dataloader 만들기
class ChatDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        # 질문은 encoder 입력
        encoder_input = sample['question_ids']

        # 답변은 decoder 입력 / target 으로 한 칸씩 나눔
        answer_ids = sample['answer_ids']
        decoder_input = answer_ids[:-1]   # 마지막 <EOS> 제외
        decoder_target = answer_ids[1:]
        return {
            'encoder_input': torch.tensor(encoder_input, dtype=torch.long),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }
chat_dataset = ChatDataset(tokenized_data)
dataloader = DataLoader(chat_dataset, batch_size=32, shuffle=True)

In [36]:
# Encoder 모델 구현
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx, embedding_matrix=None):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
    if embedding_matrix is not None:
      self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

In [37]:
# Decoder 모델 구현
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, (h_s, c_s)

In [38]:
# Seq2Seq 모델 구현
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    # h_s, c_s를 튜플이 아닌 개별 인자로 전달합니다.
    output, (h_s, c_s) = self.decoder(target, h_s, c_s)
    return output

In [17]:
vocab_size = len(word2idx)

encoder = Encoder(
    vocab_size,
    EMBEDDING_DIM,
    LATENT_DIM,
    pad_idx=word2idx['<PAD>'],
    embedding_matrix=None
)

decoder = Decoder(
    vocab_size,
    EMBEDDING_DIM,
    LATENT_DIM,
    pad_idx=word2idx['<PAD>']
)

model = Seq2Seq(encoder, decoder)
model

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(13870, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(13870, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=13870, bias=True)
  )
)

In [41]:
train_loader = DataLoader(chat_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(chat_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [20]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 5

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):

    model.train()
    train_loss, train_correct, train_tokens = 0, 0, 0

    for batch in train_loader:

        enc_inputs = batch['encoder_input']
        dec_inputs = batch['decoder_input']
        dec_targets = batch['decoder_target']

        optimizer.zero_grad()

        output = model(enc_inputs, dec_inputs)

        output = output.view(-1, output.size(-1))
        dec_targets = dec_targets.view(-1)

        loss = criterion(output, dec_targets)
        loss.backward()
        optimizer.step()

        preds = output.argmax(dim=-1)

        train_loss += loss.detach().item()

        mask = dec_targets != 0
        correct = (preds == dec_targets) & mask

        train_correct += correct.sum().item()
        train_tokens += mask.sum().item()

    train_loss /= len(train_loader)
    train_acc = train_correct / train_tokens

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # validation
    model.eval()
    val_loss, val_correct, val_tokens = 0, 0, 0

    with torch.no_grad():
        for batch in val_loader:

            enc_inputs = batch['encoder_input']
            dec_inputs = batch['decoder_input']
            dec_targets = batch['decoder_target']

            output = model(enc_inputs, dec_inputs)

            output = output.view(-1, output.size(-1))
            dec_targets = dec_targets.view(-1)

            loss = criterion(output, dec_targets)

            preds = output.argmax(dim=-1)

            val_loss += loss.detach().item()

            mask = dec_targets != 0
            correct = (preds == dec_targets) & mask

            val_correct += correct.sum().item()
            val_tokens += mask.sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens

    val_losses.append(val_loss)
    val_accs.append(val_acc)
    # 3분 1에폭
    print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

Epoch 1/5 TrainLoss=5.9186 TrainAcc=0.2173 ValLoss=5.3749 ValAcc=0.2348
Epoch 2/5 TrainLoss=5.1185 TrainAcc=0.2576 ValLoss=4.3946 ValAcc=0.3015
Epoch 3/5 TrainLoss=4.2235 TrainAcc=0.3264 ValLoss=3.4745 ValAcc=0.4039
Epoch 4/5 TrainLoss=3.4494 TrainAcc=0.4178 ValLoss=2.8483 ValAcc=0.5090
Epoch 5/5 TrainLoss=2.8937 TrainAcc=0.5003 ValLoss=2.4442 ValAcc=0.5784


In [21]:
# 모델 저장
torch.save(model.state_dict(), 'seq2seq_chatbot.pth')

# 모델 불러오기
model.load_state_dict(torch.load('seq2seq_chatbot.pth'))



<All keys matched successfully>

In [ ]:
# 챗봇 구현
# 모델 추론 + while True: 반복문으로 사용자 입력 받기
def translate(input_seq, model, word2idx, idx2word, max_len=20):

    model.eval()
    encoder = model.encoder
    decoder = model.decoder

    input_seq = torch.tensor([input_seq], dtype=torch.long)

    with torch.no_grad():
        hidden, cell = encoder(input_seq)

    sos_index = word2idx['<SOS>']
    eos_index = word2idx['<EOS>']
    pad_index = word2idx['<PAD>']

    output_sentence = []

    target_seq = torch.tensor([[sos_index]], dtype=torch.long)

    with torch.no_grad():
        for _ in range(max_len):

            output, (hidden, cell) = decoder(target_seq, hidden, cell)

            pred = output.argmax(-1).item()

            if pred == eos_index:
                break

            if pred != pad_index:
                word = idx2word.get(pred, '<UNK>')
                output_sentence.append(word)

            target_seq = torch.tensor([[pred]], dtype=torch.long)

    return ' '.join(output_sentence)


def sentence_to_ids(sentence):

    tokens = tokenizer(sentence)

    ids = [word2idx.get(t, word2idx['<UNK>']) for t in tokens]

    ids = pad_sequence(ids, max_q_len, word2idx['<PAD>'])

    return ids

In [28]:
question = "잠자고싶다"

input_ids = sentence_to_ids(question)

answer = translate(input_ids, model, word2idx, idx2word)

print(answer)

사랑 예의 없는 사람 이네 요.


In [39]:
# while 문으로 사용자 입력 받기

while True:
    print("챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.")
    user_input = input("사용자: ")
    if user_input.lower() in ['종료', 'exit', 'quit']:
        print("챗봇을 종료합니다.")
        break

    input_ids = sentence_to_ids(user_input)
    response = translate(input_ids, model, word2idx, idx2word)
    print(f"챗봇: {response}")

챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇: 쉬는 양제 간지러 해보여 하셨네요. 빼고
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료', 'exit', 또는 'quit'을 입력하세요.
챗봇을 종료합니다.


In [40]:
for i in range(20):
    print("Q:", preprocessed_ds[i]['question'])
    print("A:", preprocessed_ds[i]['answer'])
    print('-' * 50)

Q: 12시 땡!
A: 하루 가 또 가네요.
--------------------------------------------------
Q: 1 지망 학교 떨어졌어
A: 위로 해 드립니다.
--------------------------------------------------
Q: 3 박 4일 놀러 가고 싶다
A: 여행 은 언제나 좋죠.
--------------------------------------------------
Q: 3 박 4일 정도 놀러 가고 싶다
A: 여행 은 언제나 좋죠.
--------------------------------------------------
Q: PPL 심하네
A: 눈살 이 찌푸려지죠.
--------------------------------------------------
Q: SD 카드 망가졌어
A: 다시 새로 사는 게 마음 편해요.
--------------------------------------------------
Q: SD 카드 안 돼
A: 다시 새로 사는 게 마음 편해요.
--------------------------------------------------
Q: SNS 맞팔 왜 안 하지 ㅠㅠ
A: 잘 모르고 있을 수도 있어요.
--------------------------------------------------
Q: SNS 시간 낭비 인 거 아는데 매일 하는 중
A: 시간 을 정 하고 해보세요.
--------------------------------------------------
Q: SNS 시간 낭비 인데 자꾸 보게 됨
A: 시간 을 정 하고 해보세요.
--------------------------------------------------
Q: SNS 보면 나 만 빼고 다 행복 해보여
A: 자랑 하는 자리 니까 요.
--------------------------------------------------
Q: 가끔 궁금해
A: 그 사람 도 그럴 거 예요.
----------